# Qwen3-0.6B LoRA 微调 — 书名提取+分类

用 DeepSeek v4-pro 蒸馏训练数据, 微调 Qwen3-0.6B, 导出 Q4_K_M GGUF 部署到 RK3566.

**运行环境**: Google Colab T4 GPU (免费, ~15GB VRAM)
**预计时间**: 20-40min
**成本**: $0 (Colab 免费 + Unsloth 开源)

In [ ]:
# 安装依赖 (Unsloth 需要特定安装顺序)
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [ ]:
# 上传训练数据
from google.colab import files
print('请上传 book_extract_train.jsonl...')
uploaded = files.upload()

import json
fname = list(uploaded.keys())[0]
print(f'Uploaded: {fname}')

In [ ]:
# 检查训练数据
import json
from collections import Counter

data = []
with open('book_extract_train.jsonl') as f:
    for line in f:
        line = line.strip()
        if line and '"messages"' in line:
            data.append(json.loads(line))

print(f'训练数据: {len(data)} 条')

# 统计分类分布
cat_counts = Counter()
for d in data:
    msgs = d['messages']
    last = msgs[-1]['content']
    if '|' in last:
        # 提取 <think>\n\n</think>\n\n 后面的内容
        parts = last.rsplit('|', 1)
        if len(parts) == 2:
            cat = parts[1].strip()
            cat_counts[cat] += 1

print('\n分类分布:')
for cat, n in cat_counts.most_common():
    print(f'  {cat}: {n} ({100*n/len(data):.1f}%)')

# 随机看几条
import random
print('\n随机样本:')
for d in random.sample(data, min(3, len(data))):
    msgs = d['messages']
    user = msgs[1]['content']
    assistant = msgs[2]['content'].split('</think>\n\n')[-1] if '</think>' in msgs[2]['content'] else msgs[2]['content']
    print(f'  {user!r} → {assistant!r}')

In [ ]:
# 加载模型 (QLoRA 4-bit)
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print(f'Model loaded')

In [ ]:
# 配置 LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.1f}%)')

In [ ]:
# 格式化数据集
from datasets import Dataset

dataset = Dataset.from_list(data)

def format_func(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(format_func, batched=True)
print(f'Dataset: {len(dataset)} samples')
print(f'Sample length: {len(dataset[0]["text"])} chars')

In [ ]:
# 训练
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        save_steps=500,
        output_dir="./qwen3-book-lora",
        optim="adamw_8bit",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        seed=42,
    ),
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# 测试推理 (无思考模式)
FastLanguageModel.for_inference(model)

test_cases = [
    "语音：额那个三体",
    "语音：物联网可能吧控制技术",
    "语音：帮我录入活着这本书",
    "语音：我想加一个小王子",
    "语音：就那个百年孤独吧应该是",
    "语音：um Harry Potter and the Philosopher's Stone please",
    "语音：帮我add一下Python编程从入门到实践",
    "语音：明朝那个那些事儿",
]

SYSTEM_PROMPT = (
    "你是书籍提取助手。从语音识别文本中提取书名和分类，只输出一行，例如：三体|文学小说。"
    "可用分类：文学小说|少儿教育|人文社科|科技生活。不要输出其他内容。"
)

print("=" * 50)
for inp in test_cases:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": inp},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    inputs = inputs.to("cuda")
    
    outputs = model.generate(
        inputs, max_new_tokens=20, temperature=0, do_sample=False)
    
    result = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True).strip()
    print(f'{inp}\n  → {result}\n')

In [ ]:
# 保存 LoRA adapter
model.save_pretrained("qwen3-book-lora-adapter")
tokenizer.save_pretrained("qwen3-book-lora-adapter")
print('LoRA adapter saved to qwen3-book-lora-adapter/')

In [ ]:
# 合并 + 导出 GGUF Q4_K_M
print('Merging LoRA into base model...')
model.save_pretrained_merged("qwen3-book-merged", tokenizer, save_method="merged_16bit")

print('\nExporting GGUF Q4_K_M (~400MB)...')
model.save_pretrained_gguf(
    "qwen3-book-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

!ls -lh qwen3-book-gguf/

In [ ]:
# 下载 GGUF
from google.colab import files
import glob
for f in glob.glob('qwen3-book-gguf/*.gguf'):
    print(f'Downloading {f}...')
    files.download(f)

## 部署到 RK3566

```bash
# 1. 拷贝到板子
scp qwen3-book-q4_k_m.gguf root@192.168.106.26:/userdata/voice_app/models/

# 2. 修改 llm_engine.py
# MODEL_PATH = "/userdata/voice_app/models/qwen3-book-q4_k_m.gguf"

# 3. 测试
cd /userdata/voice_app
python3 -c "
from llm_engine import extract_book_info
print(extract_book_info('物联网可能吧控制技术'))
"
```

## 重新微调 (效果不好时)

直接在 Colab 重新跑这个 Notebook:
- 调整 `num_train_epochs` (增加=更多训练, 减少=防过拟合)
- 调整训练数据: 增加中文比例 / 调整噪声类型分布
- 每次微调独立, 不会累积偏差

```python
# 微调超参数建议:
num_train_epochs=3   # 默认, 欠拟合→5, 过拟合→2
learning_rate=2e-4   # 默认, 不稳定→1e-4
r=16                 # 默认, 更复杂→32, 更简单→8
```